In [1]:
# Instala as versões corretas e sincronizadas das bibliotecas
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes torchao

In [2]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
# =======================================================================
# PASSO 1 — Modelo e Tokenizador
# =======================================================================
model_id = "microsoft/bitnet-b1.58-2B-4T-bf16"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

In [4]:
# =======================================================================
# PASSO 2 — LoRA (idêntico ao QMSum e MediaSum)
# =======================================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 7,987,200 || all params: 2,420,807,680 || trainable%: 0.3299


In [5]:
# =======================================================================
# PASSO 3 — Dataset
# =======================================================================
dataset = load_dataset("knkarthick/samsum")

def aplicar_template(exemplo):
    texto_formatado = f"### Dialogue:\n{exemplo['dialogue']}\n\n### Summary:\n{exemplo['summary']}{tokenizer.eos_token}"
    return {"text": texto_formatado}

dataset_treino = dataset["train"].map(aplicar_template)
dataset_val = dataset["validation"].map(aplicar_template)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [6]:
import trl, transformers, peft
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

trl: 1.5.1
transformers: 5.10.2
peft: 0.19.1


In [7]:
# =======================================================================
# PASSO 4 — Treinamento
# =======================================================================
training_args = SFTConfig(
    output_dir="./resultados_samsum",
    max_length=1024,                  # Nova nomenclatura no trl 1.5.1
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=140,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,
    gradient_checkpointing=True,
    report_to="none",
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_treino,
    eval_dataset=dataset_val,
    args=training_args
    # peft_config removido — LoRA já aplicado no Passo 2
)

print("Iniciando treinamento SAMSum...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/14731 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14731 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/818 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/818 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Iniciando treinamento SAMSum...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
140,2.442037,2.329358,2.306342,390924.000000,0.506755
280,2.296836,2.284837,2.279665,790347.000000,0.511976
420,2.275629,2.267767,2.232077,1175794.000000,0.513740
560,2.219201,2.257527,2.237793,1564082.000000,0.514809
700,2.262555,2.251369,2.214327,1949309.000000,0.515634
840,2.223468,2.246277,2.225924,2343378.000000,0.516558
980,2.233941,2.243098,2.199445,2741804.000000,0.516873
1120,2.227749,2.240489,2.206072,3126291.000000,0.517740
1260,2.236541,2.237927,2.221107,3517927.000000,0.517615
1400,2.253600,2.235828,2.198304,3914024.000000,0.518199


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
140,2.442037,2.329358,2.306342,390924.000000,0.506755
280,2.296836,2.284837,2.279665,790347.000000,0.511976
420,2.275629,2.267767,2.232077,1175794.000000,0.513740
560,2.219201,2.257527,2.237793,1564082.000000,0.514809
700,2.262555,2.251369,2.214327,1949309.000000,0.515634
840,2.223468,2.246277,2.225924,2343378.000000,0.516558
980,2.233941,2.243098,2.199445,2741804.000000,0.516873
1120,2.227749,2.240489,2.206072,3126291.000000,0.517740
1260,2.236541,2.237927,2.221107,3517927.000000,0.517615
1400,2.253600,2.235828,2.198304,3914024.000000,0.518199


TrainOutput(global_step=2763, training_loss=2.245603903799095, metrics={'train_runtime': 12107.3478, 'train_samples_per_second': 3.65, 'train_steps_per_second': 0.228, 'total_flos': 1.6502894790586368e+17, 'train_loss': 2.245603903799095, 'epoch': 3.0})

In [8]:
# =======================================================================
# PASSO 5 — Salvar modelo e gráficos
# =======================================================================
pasta_salvamento = "./modelo_final_samsum"
os.makedirs(pasta_salvamento, exist_ok=True)
trainer.model.save_pretrained(pasta_salvamento)
print(f"Pesos LoRA salvos em: {pasta_salvamento}")

history = pd.DataFrame(trainer.state.log_history)
history.to_csv(f"{pasta_salvamento}/historico_treinamento.csv", index=False)
history.to_json(f"{pasta_salvamento}/historico_treinamento.json", orient="records", indent=4)

train_df = history[history['loss'].notna()]
eval_df = history[history['eval_loss'].notna()]

plt.figure(figsize=(10, 5))
plt.plot(train_df['step'], train_df['loss'], label='Train Loss', color='tab:blue')
if not eval_df.empty:
    plt.plot(eval_df['step'], eval_df['eval_loss'], label='Eval Loss', color='tab:orange')

plt.title('BitNet b1.58 + LoRA (SAMSum) - Curva de Aprendizado')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.savefig(f"{pasta_salvamento}/loss_curve.png", dpi=300, bbox_inches='tight')
plt.savefig(f"{pasta_salvamento}/loss_curve.pdf", bbox_inches='tight')
plt.close()
print("Gráficos salvos com sucesso!")

Pesos LoRA salvos em: ./modelo_final_samsum
Gráficos salvos com sucesso!


In [9]:
# =======================================================================
# PASSO 6 — Consolidar e zipar tudo para backup local
# =======================================================================
import shutil
import os

pasta_backup = "./backup_samsum_completo"
os.makedirs(pasta_backup, exist_ok=True)

# Copia resultados (gráficos, CSV, JSON, checkpoints)
shutil.copytree("./resultados_samsum", f"{pasta_backup}/resultados_samsum", dirs_exist_ok=True)

# Copia pesos do LoRA
shutil.copytree("./modelo_final_samsum", f"{pasta_backup}/modelo_final_samsum", dirs_exist_ok=True)

# Zipa tudo em um único arquivo
shutil.make_archive("./backup_samsum_completo", "zip", pasta_backup)

print("Backup gerado com sucesso: backup_samsum_completo.zip")
print(f"Tamanho: {os.path.getsize('./backup_samsum_completo.zip') / (1024**2):.1f} MB")

Backup gerado com sucesso: backup_samsum_completo.zip
Tamanho: 114.5 MB


In [10]:
from google.colab import files
files.download("./backup_samsum_completo.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>